
# 03. Traditional Machine Learning

## Objective

Develop and evaluate traditional machine learning models for peptide MIC regression using sequence-derived features.

## Models

- Random Forest (RF)
- Support Vector Regression (SVR)
- Decision Tree (DT)
- K-Nearest Neighbors (KNN)

## Sequence Features

- AAC
- DPC
- CTDC
- CTDT
- CTDD
- QSOrder

## Workflow

1. Load EC and SA datasets.
2. Extract and save sequence features.
3. Load predefined training/validation splits.
4. Optimize model hyperparameters.
5. Train and select the best models.
6. Evaluate independent test datasets.
7. Save models, predictions, and performance metrics.

## Experimental Settings

- Random seeds: 42, 43, 44, 45, 46
- Regression target: pMIC
- Model selection: Validation MSE
- Evaluation: MSE, RMSE, MAE, R², PCC, KTC
- XGBoost is excluded.

All models use the same predefined dataset splits and evaluation protocol.


In [1]:

import pandas as pd

from configs.common import DATA_DIR, RANDOM_SEEDS
from src.data import load_dataset
from src.seed import set_seed

# Initialize random seed
set_seed(RANDOM_SEEDS[0])

# Load datasets
datasets = {
    f"{target}_{split}": load_dataset(target, split)
    for target in ["EC", "SA"]
    for split in ["Train", "Test"]
}

# Check dataset structure
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

# Verify validation split files
for target in ["EC", "SA"]:
    df = datasets[f"{target}_Train"]

    for seed in RANDOM_SEEDS:
        path = DATA_DIR / "splits" / f"{target}_seed{seed}.csv"

        if not path.is_file():
            raise FileNotFoundError(path)

        split_df = pd.read_csv(path)

        assert len(split_df) == len(df)
        assert split_df["row_index"].tolist() == list(range(len(df)))
        assert set(split_df["subset"]) == {"train", "valid"}

        train_seq = set(
            df.loc[split_df["subset"] == "train", "sequence"]
        )
        valid_seq = set(
            df.loc[split_df["subset"] == "valid", "sequence"]
        )

        assert train_seq.isdisjoint(valid_seq)

print("\n[OK] All datasets and validation splits verified.")


EC_Train: (7583, 8)
EC_Test: (843, 8)
SA_Train: (6592, 8)
SA_Test: (733, 8)

[OK] All datasets and validation splits verified.


In [2]:

from src.features import extract_features, FEATURE_DIMENSIONS
from configs.common import DATA_DIR

# Feature output directory
FEATURE_DIR = DATA_DIR / "features"

# Extract AAC and DPC
for target in ["EC", "SA"]:

    output_dir = FEATURE_DIR / target
    output_dir.mkdir(parents=True, exist_ok=True)

    for split in ["Train", "Test"]:

        df = datasets[f"{target}_{split}"]

        for feature_type in ["AAC", "DPC"]:

            # Extract features
            feature_df = extract_features(
                df, feature_type
            )
            # Save features
            output_path = (
                output_dir / f"{feature_type}_{split}.csv"
            )

            feature_df.to_csv(
                output_path,
                index=False
            )

            print(
                f"[OK] {target} {split} "
                f"{feature_type}: {feature_df.shape}"
            )

print("\\n[OK] AAC and DPC extraction completed.")


[OK] EC Train AAC: (7583, 22)
[OK] EC Train DPC: (7583, 402)
[OK] EC Test AAC: (843, 22)
[OK] EC Test DPC: (843, 402)
[OK] SA Train AAC: (6592, 22)
[OK] SA Train DPC: (6592, 402)
[OK] SA Test AAC: (733, 22)
[OK] SA Test DPC: (733, 402)
\n[OK] AAC and DPC extraction completed.


In [3]:
###### TEST

import importlib
import src.features as features

# Reload updated module
importlib.reload(features)

# Test with 5 peptides
test_df = datasets["EC_Train"].head(5).copy()

descriptors = ["CTDC", "CTDT", "CTDD", "QSOrder"]

# Extract features
for descriptor in descriptors:
    result = features.extract_ifeature(
        test_df, descriptor
    )

    print(
        f"{descriptor}: "
        f"{result.shape[0]} samples, "
        f"{result.shape[1] - 2} features"
    )

    display(result.iloc[:3, :5])

CTDC: 5 samples, 39 features


,sequence,pMIC,CTDC_hydrophobicity_PRAM900101.G1,CTDC_hydrophobicity_PRAM900101.G2,CTDC_hydrophobicity_PRAM900101.G3
0,AAAAAAAIKMLMDLVNERIMALNKKAKK,5.000000,0.357143,0.321429,0.321429
1,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,5.749537,0.325000,0.375000,0.300000
2,AAGKGLVSNLLEK,4.609065,0.307692,0.384615,0.307692


CTDT: 5 samples, 39 features


,sequence,pMIC,CTDT_hydrophobicity_PRAM900101.Tr1221,CTDT_hydrophobicity_PRAM900101.Tr1331,CTDT_hydrophobicity_PRAM900101.Tr2332
0,AAAAAAAIKMLMDLVNERIMALNKKAKK,5.000000,0.074074,0.259259,0.111111
1,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,5.749537,0.256410,0.205128,0.230769
2,AAGKGLVSNLLEK,4.609065,0.250000,0.166667,0.166667


CTDD: 5 samples, 195 features


,sequence,pMIC,CTDD_hydrophobicity_PRAM900101.1.residue0,CTDD_hydrophobicity_PRAM900101.1.residue25,CTDD_hydrophobicity_PRAM900101.1.residue50
0,AAAAAAAIKMLMDLVNERIMALNKKAKK,5.000000,32.142857,46.428571,64.285714
1,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,5.749537,12.500000,32.500000,52.500000
2,AAGKGLVSNLLEK,4.609065,30.769231,30.769231,69.230769


QSOrder: 5 samples, 46 features


,sequence,pMIC,QSOrder_Schneider.Xr.A,QSOrder_Schneider.Xr.R,QSOrder_Schneider.Xr.N
0,AAAAAAAIKMLMDLVNERIMALNKKAKK,5.000000,3.832523,0.425836,0.851672
1,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,5.749537,1.133396,0.850047,0.566698
2,AAGKGLVSNLLEK,4.609065,1.262328,0.000000,0.631164


In [4]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

from configs.common import DATA_DIR
from src.features import extract_features, extract_ifeature

# Feature configuration
FEATURES = [
    "AAC", "DPC",
    "CTDC", "CTDT", "CTDD", "QSOrder"
]

FEATURE_DIR = DATA_DIR / "features"

# Extract and save features
tasks = [
    (target, split, feature)
    for target in ["EC", "SA"]
    for split in ["Train", "Test"]
    for feature in FEATURES
]

with tqdm(tasks, desc="Feature Extraction", unit="task") as pbar:
    for target, split, feature in pbar:

        output_dir = FEATURE_DIR / target
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / f"{feature}_{split}.csv"

        # Reuse existing features
        if output_path.exists():
            pbar.set_postfix_str(f"{target} {split} {feature} | cached")
            continue

        df = datasets[f"{target}_{split}"]

        # Extract features
        if feature in ["AAC", "DPC"]:
            result = extract_features(df, feature)
        else:
            result = extract_ifeature(df, feature)

        # Save features
        result.to_csv(output_path, index=False)

        pbar.set_postfix_str(
            f"{target} {split} {feature} | {result.shape[1] - 2} features"
        )

print("\n[OK] All features saved.")


c:\Users\jinwo\anaconda3\envs\bert-ampep60\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Feature Extraction: 100%|██████████| 24/24 [00:12<00:00,  1.97task/s, SA Test QSOrder | 46 features] 


[OK] All features saved.


In [5]:

import numpy as np
import pandas as pd

# Verify saved features
summary = []

for target in ["EC", "SA"]:
    for split in ["Train", "Test"]:
        for feature in FEATURES:

            path = FEATURE_DIR / target / f"{feature}_{split}.csv"
            df = pd.read_csv(path)

            expected = datasets[f"{target}_{split}"]

            assert len(df) == len(expected)
            assert df["sequence"].tolist() == expected["sequence"].tolist()
            assert np.allclose(df["pMIC"], expected["pMIC"])

            summary.append({
                "Dataset": target,
                "Split": split,
                "Feature": feature,
                "Samples": len(df),
                "Dimensions": df.shape[1] - 2
            })

summary_df = pd.DataFrame(summary)

display(summary_df)

print("\n[OK] All 24 feature files verified.")


,Dataset,Split,Feature,Samples,Dimensions
0,EC,Train,AAC,7583,20
1,EC,Train,DPC,7583,400
2,EC,Train,CTDC,7583,39
3,EC,Train,CTDT,7583,39
4,EC,Train,CTDD,7583,195
5,EC,Train,QSOrder,7583,46
6,EC,Test,AAC,843,20
7,EC,Test,DPC,843,400
8,EC,Test,CTDC,843,39
9,EC,Test,CTDT,843,39



[OK] All 24 feature files verified.
